# Copernicus openEO — Earth Surface Features (Google Colab)

**Input:** CSV uploaded from your computer containing `site_id,latitude_deg,longitude_deg` (or `latitude,longitude`).  
**Output:** `openeo_features.csv` automatically downloaded from Colab.

Processing is performed on the Copernicus Data Space openEO backend. Satellite raster data are **not** downloaded. Only aggregated JSON results are returned.

**Scientific interpretation:** `vegetation_cover_pct` is an **NDVI-derived greenness proxy**, not direct fractional canopy cover. `soil_moisture_fraction` is a **relative Sentinel-1 VV wetness proxy**, not calibrated volumetric soil moisture. WorldCover is land cover, not geomorphology.

Run the cells sequentially. Batch processing may incur backend quotas or fail if a collection/process configuration changes.

In [ ]:
!pip -q install -U openeo pandas numpy shapely pyproj

## 1. Upload your input CSV
Run this cell and select your CSV file. No need to save it in Google Drive.

In [ ]:
from google.colab import files
from pathlib import Path
import io
import pandas as pd

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a CSV file.")
filename = next((name for name in uploaded if name.lower().endswith('.csv')), None)
if filename is None:
    raise ValueError("No CSV file was uploaded.")
INPUT_FILE = filename
preview = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"Uploaded {filename}: {len(preview)} rows")
display(preview.head())

## 2. Parameters
The time window uses 2024 imagery. WorldCover is a static 2021 classification. For initial testing, use 1–3 coordinates to control processing costs.

In [ ]:
START_DATE = "2024-01-01"
END_DATE = "2025-01-01"  # openEO temporal end
BUFFER_M = 100
GROUP_GRID_DEG = 2
MAX_SITES_PER_GROUP = 10
OUTPUT_FILE = "openeo_features.csv"
ENDPOINT = "https://openeo.dataspace.copernicus.eu"

## 3. Processing functions
The code groups nearby coordinates so it does not create a planet-wide bounding box. For each group it aggregates WorldCover classes, cloud-masked Sentinel-2 NDVI and Sentinel-1 VV backscatter into small JSON results.

In [ ]:
import json
import logging
import math
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import openeo
from shapely.geometry import Point, mapping
from shapely.ops import transform
from pyproj import CRS, Transformer

WORKDIR = Path('/content/openeo_json')
WORKDIR.mkdir(parents=True, exist_ok=True)
LAND_COVER = {
    10: 'Forest', 20: 'Shrubland', 30: 'Grassland', 40: 'Cropland',
    50: 'Built-up', 60: 'Bare/Sparse vegetation', 70: 'Snow and Ice',
    80: 'Permanent Water', 90: 'Wetland', 95: 'Mangroves',
    100: 'Moss and Lichen'
}
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('openeo')


def load_sites(path):
    df = pd.read_csv(path, dtype={'site_id': str})
    df = df.rename(columns={'latitude':'latitude_deg', 'longitude':'longitude_deg',
                            'lat':'latitude_deg', 'lon':'longitude_deg', 'lng':'longitude_deg'})
    if 'site_id' not in df.columns:
        df.insert(0, 'site_id', [f'SITE_{i:04}' for i in range(1, len(df)+1)])
    required = ['site_id','latitude_deg','longitude_deg']
    absent = [c for c in required if c not in df]
    if absent: raise ValueError(f'Missing columns: {absent}')
    df = df[required].copy()
    for c in ('latitude_deg','longitude_deg'):
        df[c] = pd.to_numeric(df[c], errors='coerce')
    if df['site_id'].isna().any() or df['site_id'].duplicated().any():
        raise ValueError('site_id must be nonempty and unique')
    valid = df.latitude_deg.between(-85,85) & df.longitude_deg.between(-180,180)
    if not valid.all():
        raise ValueError('Invalid coordinates or latitude beyond supported ±85°')
    if len(df) == 0: raise ValueError('CSV file is empty')
    return df.reset_index(drop=True)


def buffer_geometry(lon, lat):
    local = CRS.from_proj4(f'+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m +no_defs')
    proj = Transformer.from_crs(local, 'EPSG:4326', always_xy=True)
    return mapping(transform(proj.transform, Point(0,0).buffer(BUFFER_M, resolution=12)))


def geometries_and_bbox(df):
    features, coords = [], []
    for row in df.itertuples(index=False):
        geom = buffer_geometry(row.longitude_deg, row.latitude_deg)
        features.append({'type':'Feature','properties':{'site_id':row.site_id},'geometry':geom})
        coords.extend(geom['coordinates'][0])
    lon = [p[0] for p in coords]
    lat = [p[1] for p in coords]
    if max(lon)-min(lon) > 180:
        raise ValueError('Antimeridian-crossing sites require separate handling')
    bbox = {'west':max(-180,min(lon)-0.002), 'east':min(180,max(lon)+0.002),
            'south':max(-90,min(lat)-0.002), 'north':min(90,max(lat)+0.002), 'crs':'EPSG:4326'}
    return {'type':'FeatureCollection','features':features}, bbox


def group_sites(df):
    groups=defaultdict(list)
    for row in df.to_dict('records'):
        k=(math.floor(row['latitude_deg']/GROUP_GRID_DEG),
           math.floor(row['longitude_deg']/GROUP_GRID_DEG))
        groups[k].append(row)
    for chunk in groups.values():
        for i in range(0,len(chunk),MAX_SITES_PER_GROUP):
            yield pd.DataFrame(chunk[i:i+MAX_SITES_PER_GROUP])


def worldcover_cube(conn,bbox):
    collection='ESA_WORLDCOVER_10M_2021_V2'
    meta=conn.describe_collection(collection)
    band_names=meta.get('cube:dimensions',{}).get('bands',{}).get('values',[])
    if len(band_names)!=1: raise RuntimeError(f'WorldCover has unexpected bands: {band_names}')
    cube=conn.load_collection(collection,spatial_extent=bbox,
                              temporal_extent=['2021-01-01','2022-01-01'])
    base=cube.band(band_names[0])
    result=None
    for code in LAND_COVER:
        layer=(base==code).apply(lambda x:x*1.0).rename_labels(dimension='bands',target=[f'class_{code}'])
        result=layer if result is None else result.merge_cubes(layer)
    return result


def ndvi_cube(conn,bbox):
    cube=conn.load_collection('SENTINEL2_L2A',spatial_extent=bbox,
                              temporal_extent=[START_DATE,END_DATE],
                              bands=['B04','B08','SCL'],max_cloud_cover=80)
    red,nir,scl=cube.band('B04'),cube.band('B08'),cube.band('SCL')
    invalid=(scl==0)|(scl==1)|(scl==3)|(scl==8)|(scl==9)|(scl==10)|(scl==11)
    ndvi=((nir-red)/(nir+red+0.0001)).mask(invalid)
    return ndvi.reduce_temporal('median')


def sar_cube(conn,bbox):
    cube=conn.load_collection('SENTINEL1_GRD',spatial_extent=bbox,
                              temporal_extent=[START_DATE,END_DATE],bands=['VV'],
                              properties={'sat:orbit_state':lambda x:x=='ASCENDING'})
    return cube.sar_backscatter(coefficient='sigma0-ellipsoid')


def execute_json(cube,geom,name,gid):
    path=WORKDIR/f'group_{gid}_{name}.json'
    aggregated=cube.aggregate_spatial(geometries=geom,reducer='mean')
    aggregated.execute_batch(outputfile=str(path),out_format='JSON',
                             title=f'Surface features {name}, group {gid}')
    with path.open(encoding='utf8') as f:
        return json.load(f)


def to_array(data,n_sites,n_bands):
    """Standard openEO vector-cube JSON: date -> polygons -> bands."""
    records=[]
    for date in sorted(data):
        polygons=data[date]
        if len(polygons)!=n_sites:
            raise ValueError(f'At {date}, expected {n_sites} sites; got {len(polygons)}')
        row=[]
        for values in polygons:
            if values is None: values=[None]*n_bands
            if len(values)!=n_bands:
                raise ValueError(f'At {date}, expected {n_bands} bands; got {len(values)}')
            row.append([float(v) if v is not None else np.nan for v in values])
        records.append(row)
    return np.array(records,dtype=float) if records else np.empty((0,n_sites,n_bands))


def median_finite(values):
    v=np.asarray(values,dtype=float)
    v=v[np.isfinite(v)]
    return float(np.median(v)) if len(v) else np.nan


def combine_result(wc_raw,ndvi_raw,sar_raw,ids):
    n=len(ids)
    wc=to_array(wc_raw,n,len(LAND_COVER))
    nv=to_array(ndvi_raw,n,1)
    sar=to_array(sar_raw,n,1)
    rows=[]
    codes=list(LAND_COVER)
    for i,site_id in enumerate(ids):
        item={'site_id':site_id}
        scores=np.array([median_finite(wc[:,i,j]) for j in range(len(codes))])
        if np.isfinite(scores).any():
            k=int(np.nanargmax(scores))
            item.update(terrain_type=LAND_COVER[codes[k]],terrain_code=codes[k],
                        terrain_dominance_pct=round(float(scores[k])*100,2))
        else:
            item.update(terrain_type='Unknown',terrain_code=pd.NA,terrain_dominance_pct=np.nan)
        ndvi=median_finite(nv[:,i,0])
        item['ndvi_median']=round(ndvi,4) if np.isfinite(ndvi) else np.nan
        item['vegetation_cover_pct']=(round(float(np.clip((ndvi-0.2)/0.6,0,1))*100,2)
                                      if np.isfinite(ndvi) else np.nan)
        vv=sar[:,i,0]
        vv=vv[np.isfinite(vv)&(vv>0)]
        item['sar_observation_count']=int(len(vv))
        item['sar_vv_median_db']=(round(float(10*np.log10(np.median(vv))),3)
                                  if len(vv) else np.nan)
        wetness=np.nan
        if len(vv)>=3:
            db=10*np.log10(vv)
            lo,hi=np.percentile(db,[10,90])
            if hi-lo>0.01: wetness=float(np.clip((np.median(db)-lo)/(hi-lo),0,1))
        item['soil_moisture_fraction']=round(wetness,4) if np.isfinite(wetness) else np.nan
        rows.append(item)
    return pd.DataFrame(rows)


def run_pipeline(input_file=INPUT_FILE, output_file=OUTPUT_FILE):
    sites=load_sites(input_file)
    print(f'Loaded {len(sites)} site(s)')
    conn=openeo.connect(ENDPOINT)
    conn.authenticate_oidc()
    needed={'ESA_WORLDCOVER_10M_2021_V2','SENTINEL2_L2A','SENTINEL1_GRD'}
    missing=needed-set(conn.list_collection_ids())
    if missing: raise RuntimeError(f'Unavailable CDSE collections: {sorted(missing)}')
    groups=list(group_sites(sites))
    parts=[]
    for gid,grp in enumerate(groups,1):
        print(f'Processing group {gid}/{len(groups)} with {len(grp)} site(s)')
        geom,bbox=geometries_and_bbox(grp)
        wc=execute_json(worldcover_cube(conn,bbox),geom,'worldcover',gid)
        nv=execute_json(ndvi_cube(conn,bbox),geom,'ndvi',gid)
        sar=execute_json(sar_cube(conn,bbox),geom,'sar',gid)
        parts.append(combine_result(wc,nv,sar,grp.site_id.tolist()))
    out=sites.merge(pd.concat(parts,ignore_index=True),on='site_id',how='left',validate='one_to_one')
    out['terrain_type']=out['terrain_type'].fillna('Unknown').astype('string')
    out['terrain_code']=pd.to_numeric(out['terrain_code'],errors='coerce').astype('Int64')
    for c in ['terrain_dominance_pct','vegetation_cover_pct','ndvi_median',
              'soil_moisture_fraction','sar_vv_median_db','sar_observation_count']:
        out[c]=pd.to_numeric(out[c],errors='coerce')
    out.to_csv(output_file,index=False,na_rep='')
    return out


## 4. Run the extraction
This cell starts cloud batch jobs. The notebook waits for them to finish; incomplete groups produce an exception rather than silently inventing missing observations.

In [ ]:
results = run_pipeline()
display(results.head(15))
print(f"Saved {len(results)} rows to {OUTPUT_FILE}")

## 5. Download the result to your computer
Colab opens a browser download for the finished CSV.

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

### Notes
- `soil_moisture_fraction`: relative index normalized **within each site over time**, not a globally comparable soil moisture fraction. Keep `sar_vv_median_db` for a more consistent raw indicator, noting sensitivity to geometry, vegetation and surface roughness.
- `vegetation_cover_pct`: NDVI proxy created by linear rescaling (0.2→0%, 0.8→100%). Do not interpret as directly measured vegetation cover.
- `terrain_type`: dominant WorldCover 2021 land-cover category. `Bare/Sparse vegetation` does not automatically mean desert.
- This notebook has **not** been executed against the live CDSE backend here. The backend may require changes to collection-specific bands, properties or processing support; test on one site first.
- JSON intermediate outputs remain in `/content/openeo_json/` during the Colab session.